In [ ]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.5)
  from scipy.sparse import csr_matrix, issparse


Device: cuda


In [ ]:
base_path = "/content/drive/MyDrive/data/data/non_cuda"

datasets_local = {
    "airline": pd.read_csv(os.path.join(base_path, "airline.tsv"), sep="\t", encoding="latin1"),
    "eco_news": pd.read_csv(os.path.join(base_path, "eco_news.tsv"), sep="\t", encoding="latin1"),
    "glo_warm": pd.read_csv(os.path.join(base_path, "glo_warm.tsv"), sep="\t", encoding="latin1"),
    "text_emo": pd.read_csv(os.path.join(base_path, "text_emo.tsv"), sep="\t", encoding="latin1"),
}

for name, df in datasets_local.items():
    print("====", name, "====")
    print(df.shape)
    print(df.head(2))
    print(df["label"].value_counts().head(), "\n")


==== airline ====
(14640, 2)
                                                text     label
0                @VirginAmerica What @dhepburn said.   neutral
1  @VirginAmerica plus you've added commercials t...  positive
label
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64 

==== eco_news ====
(8000, 2)
                                                text label
0  NEW YORK -- Yields on most certificates of dep...   yes
1  The Wall Street Journal Online</br></br>The Mo...    no
label
no          6571
yes         1420
not sure       9
Name: count, dtype: int64 

==== glo_warm ====
(4225, 2)
                                                text label
0  Global warming report urges governments to act...   Yes
1  Fighting poverty and global warming in Africa ...   Yes
label
Y      2554
N      1053
Yes     557
No       61
Name: count, dtype: int64 

==== text_emo ====
(40000, 2)
                                                text    label
0  @tiffanylue i know  i w

In [ ]:
from collections import defaultdict

encoders = {}
splits = {}

def prepare_local_dataset(name, test_size=0.15, val_size=0.15):
    df = datasets_local[name].copy()
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    le = LabelEncoder()
    df["label_id"] = le.fit_transform(df["label"])
    encoders[name] = le

    X_train, X_temp, y_train, y_temp = train_test_split(
        df["text"], df["label_id"], test_size=(test_size + val_size), random_state=42, stratify=df["label_id"]
    )
    rel_val = val_size / (test_size + val_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1 - rel_val), random_state=42, stratify=y_temp
    )

    splits[name] = {
        "train": (X_train, y_train),
        "val": (X_val, y_val),
        "test": (X_test, y_test),
    }
    print(f"{name}: classes -> {list(le.classes_)}")
    for k, (X, y) in splits[name].items():
        print(f"  {k}: {len(X)} examples")
    return splits[name]

_ = prepare_local_dataset("airline")
_ = prepare_local_dataset("eco_news")
_ = prepare_local_dataset("glo_warm")
_ = prepare_local_dataset("text_emo")


airline: classes -> ['negative', 'neutral', 'positive']
  train: 10248 examples
  val: 2196 examples
  test: 2196 examples
eco_news: classes -> ['no', 'not sure', 'yes']
  train: 5600 examples
  val: 1200 examples
  test: 1200 examples
glo_warm: classes -> ['n', 'no', 'y', 'yes']
  train: 2957 examples
  val: 634 examples
  test: 634 examples
text_emo: classes -> ['anger', 'boredom', 'empty', 'enthusiasm', 'fun', 'happiness', 'hate', 'love', 'neutral', 'relief', 'sadness', 'surprise', 'worry']
  train: 28000 examples
  val: 6000 examples
  test: 6000 examples


In [ ]:
from datasets import load_dataset

hf_datasets = {}

hf_datasets["imdb"] = load_dataset("imdb")                     # binary
hf_datasets["yelp_polarity"] = load_dataset("yelp_polarity")   # binary
hf_datasets["amazon_polarity"] = load_dataset("amazon_polarity")  # binary
hf_datasets["tweet_eval_sentiment"] = load_dataset("tweet_eval", "sentiment")  # 3-way

for name, ds in hf_datasets.items():
    print("====", name, "====")
    print(ds)
    print(ds["train"][0])
    print()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

==== imdb ====
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking 

In [ ]:
def prepare_hf_dataset_for_classification(name, text_col="text", label_col="label", sample_train=None, sample_test=None):
    ds = hf_datasets[name]
    train_ds = ds["train"]
    test_ds = ds["test"] if "test" in ds else ds["validation"]

    if sample_train is not None:
        train_ds = train_ds.shuffle(seed=42).select(range(sample_train))
    if sample_test is not None:
        test_ds = test_ds.shuffle(seed=42).select(range(sample_test))

    train_df = pd.DataFrame({ "text": train_ds[text_col], "label_id": train_ds[label_col] })
    test_df = pd.DataFrame({ "text": test_ds[text_col], "label_id": test_ds[label_col] })

    le = LabelEncoder()
    le.fit(train_df["label_id"].tolist() + test_df["label_id"].tolist())

    train_df["label_id"] = le.transform(train_df["label_id"])
    test_df["label_id"] = le.transform(test_df["label_id"])

    X_train, X_val, y_train, y_val = train_test_split(
        train_df["text"], train_df["label_id"], test_size=0.15, random_state=42, stratify=train_df["label_id"]
    )

    splits[name] = {
        "train": (X_train, y_train),
        "val": (X_val, y_val),
        "test": (test_df["text"], test_df["label_id"]),
    }
    encoders[name] = le

    print(f"{name}: classes -> {list(le.classes_)}")
    for k, (X, y) in splits[name].items():
        print(f"  {k}: {len(X)} examples")
    return splits[name]

_ = prepare_hf_dataset_for_classification("imdb", sample_train=8000, sample_test=2000)


imdb: classes -> [np.int64(0), np.int64(1)]
  train: 6800 examples
  val: 1200 examples
  test: 2000 examples


In [ ]:
def run_xgboost_experiment(dataset_name, max_features=50000, n_estimators=300):
    X_train, y_train = splits[dataset_name]["train"]
    X_val, y_val = splits[dataset_name]["val"]
    X_test, y_test = splits[dataset_name]["test"]

    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        lowercase=True,
        strip_accents="unicode",
    )
    X_train_vec = tfidf.fit_transform(X_train)
    X_val_vec = tfidf.transform(X_val)
    X_test_vec = tfidf.transform(X_test)

    num_classes = len(np.unique(y_train))
    xgb = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob" if num_classes > 2 else "binary:logistic",
        eval_metric="mlogloss",
        tree_method="hist",  # change to "gpu_hist" if your XGBoost build supports GPU
        n_jobs=-1,
    )

    xgb.fit(X_train_vec, y_train, eval_set=[(X_val_vec, y_val)], verbose=True)

    def evaluate(split_name, X_vec, y_true):
        y_pred = xgb.predict(X_vec)
        acc = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, average="macro")
        print(f"{split_name} accuracy: {acc:.4f}, macro-F1: {f1_macro:.4f}")
        return y_pred

    print("=== XGBoost results on", dataset_name, "===")
    _ = evaluate("Val", X_val_vec, y_val)
    y_pred_test = evaluate("Test", X_test_vec, y_test)
    print("\nClassification report (test):")
    print(classification_report(y_test, y_pred_test, digits=4))

# examples:
run_xgboost_experiment("airline")
# run_xgboost_experiment("text_emo")
# run_xgboost_experiment("imdb")


[0]	validation_0-mlogloss:0.95539
[1]	validation_0-mlogloss:0.92507
[2]	validation_0-mlogloss:0.89697
[3]	validation_0-mlogloss:0.87170
[4]	validation_0-mlogloss:0.85093
[5]	validation_0-mlogloss:0.83289
[6]	validation_0-mlogloss:0.81684
[7]	validation_0-mlogloss:0.80133
[8]	validation_0-mlogloss:0.78799
[9]	validation_0-mlogloss:0.77604
[10]	validation_0-mlogloss:0.76491
[11]	validation_0-mlogloss:0.75387
[12]	validation_0-mlogloss:0.74450
[13]	validation_0-mlogloss:0.73527
[14]	validation_0-mlogloss:0.72709
[15]	validation_0-mlogloss:0.71981
[16]	validation_0-mlogloss:0.71315
[17]	validation_0-mlogloss:0.70615
[18]	validation_0-mlogloss:0.69999
[19]	validation_0-mlogloss:0.69292
[20]	validation_0-mlogloss:0.68755
[21]	validation_0-mlogloss:0.68263
[22]	validation_0-mlogloss:0.67804
[23]	validation_0-mlogloss:0.67321
[24]	validation_0-mlogloss:0.66936
[25]	validation_0-mlogloss:0.66490
[26]	validation_0-mlogloss:0.66076
[27]	validation_0-mlogloss:0.65771
[28]	validation_0-mlogloss:0.6

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


Device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def create_dataloaders(dataset_name, max_length=128, batch_size=64):
    X_train, y_train = splits[dataset_name]["train"]
    X_val, y_val = splits[dataset_name]["val"]
    X_test, y_test = splits[dataset_name]["test"]

    def encode_texts(texts):
        return tokenizer(
            list(texts),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

    train_enc = encode_texts(X_train)
    val_enc = encode_texts(X_val)
    test_enc = encode_texts(X_test)

    train_dataset = TextDataset(train_enc, y_train.to_numpy())
    val_dataset = TextDataset(val_enc, y_val.to_numpy())
    test_dataset = TextDataset(test_enc, y_test.to_numpy())

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    num_labels = len(np.unique(y_train))
    return train_loader, val_loader, test_loader, num_labels

train_loader_airline, val_loader_airline, test_loader_airline, num_labels_airline = create_dataloaders("airline")
num_labels_airline


3

In [ ]:
def compute_metrics_from_logits(all_logits, all_labels):
    preds = np.argmax(all_logits, axis=1)
    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, average="macro")
    return acc, f1

def evaluate_model(model, data_loader):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids=input_ids)  # RNN models ignore attention mask
            all_logits.append(logits.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    acc, f1 = compute_metrics_from_logits(all_logits, all_labels)
    return acc, f1

def train_rnn_style_model(model, train_loader, val_loader, num_epochs=4, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_acc, val_f1 = evaluate_model(model, val_loader)
        print(f"Epoch {epoch}: train_loss={total_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}")
    return model


In [ ]:
def evaluate_transformer(model, data_loader):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            all_logits.append(logits.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    acc, f1 = compute_metrics_from_logits(all_logits, all_labels)
    return acc, f1

def train_transformer_experiment(
    dataset_name,
    model_name="distilbert-base-uncased",
    epochs=3,
    max_length=128,
    batch_size=32,
    lr=2e-5,
):
    train_loader, val_loader, test_loader, num_labels = create_dataloaders(
        dataset_name, max_length=max_length, batch_size=batch_size
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
    )
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        val_acc, val_f1 = evaluate_transformer(model, val_loader)
        print(f"Epoch {epoch}: train_loss={total_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}")

    test_acc, test_f1 = evaluate_transformer(model, test_loader)
    print(f"\nTest results (Transformer {model_name}, {dataset_name}): acc={test_acc:.4f}, macro_f1={test_f1:.4f}")
    return model

transformer_airline = train_transformer_experiment("airline", epochs=5)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1: train_loss=0.6516, val_acc=0.8274, val_f1=0.7684
Epoch 2: train_loss=0.3609, val_acc=0.8329, val_f1=0.7919
Epoch 3: train_loss=0.2516, val_acc=0.8424, val_f1=0.7907
Epoch 4: train_loss=0.1719, val_acc=0.8374, val_f1=0.7914
Epoch 5: train_loss=0.1278, val_acc=0.8383, val_f1=0.7912

Test results (Transformer distilbert-base-uncased, airline): acc=0.8415, macro_f1=0.7946


In [ ]:
from datasets import load_dataset

# Add Twitter emotion dataset (6 emotions: anger, fear, joy, love, sadness, surprise)
hf_datasets["emotion"] = load_dataset("dair-ai/emotion")

# Prepare both 'emotion' and tweet_eval_sentiment in the same way as imdb
_ = prepare_hf_dataset_for_classification("emotion", text_col="text", label_col="label")
_ = prepare_hf_dataset_for_classification("tweet_eval_sentiment", text_col="text", label_col="label")

print("Available datasets in splits:")
print(list(splits.keys()))


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

emotion: classes -> [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  train: 13600 examples
  val: 2400 examples
  test: 2000 examples
tweet_eval_sentiment: classes -> [np.int64(0), np.int64(1), np.int64(2)]
  train: 38772 examples
  val: 6843 examples
  test: 12284 examples
Available datasets in splits:
['airline', 'eco_news', 'glo_warm', 'text_emo', 'imdb', 'emotion', 'tweet_eval_sentiment']


In [ ]:
exp_results = []

def add_result(dataset, model_name, val_acc, val_f1, test_acc, test_f1):
    exp_results.append({
        "dataset": dataset,
        "model": model_name,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
    })


In [ ]:
def run_xgboost_experiment_metrics(dataset_name, max_features=50000, n_estimators=300):
    X_train, y_train = splits[dataset_name]["train"]
    X_val, y_val = splits[dataset_name]["val"]
    X_test, y_test = splits[dataset_name]["test"]

    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        lowercase=True,
        strip_accents="unicode",
    )
    X_train_vec = tfidf.fit_transform(X_train)
    X_val_vec = tfidf.transform(X_val)
    X_test_vec = tfidf.transform(X_test)

    num_classes = len(np.unique(y_train))
    xgb = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob" if num_classes > 2 else "binary:logistic",
        eval_metric="mlogloss",
        tree_method="hist",  # change to "gpu_hist" if your build supports it
        n_jobs=-1,
    )

    xgb.fit(X_train_vec, y_train, eval_set=[(X_val_vec, y_val)], verbose=True)

    def eval_block(X_vec, y_true):
        y_pred = xgb.predict(X_vec)
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")
        return acc, f1

    val_acc, val_f1 = eval_block(X_val_vec, y_val)
    test_acc, test_f1 = eval_block(X_test_vec, y_test)

    print(f"[XGBoost] {dataset_name} -> val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, test_acc={test_acc:.4f}, test_f1={test_f1:.4f}")
    return val_acc, val_f1, test_acc, test_f1


In [ ]:
def run_rnn_with_metrics(dataset_name, model_type="lstm", epochs=5,
                         max_length=128, batch_size=64, embed_dim=128, hidden_dim=128):
    train_loader, val_loader, test_loader, num_labels = create_dataloaders(
        dataset_name, max_length=max_length, batch_size=batch_size
    )
    vocab_size = tokenizer.vocab_size

    if model_type == "rnn":
        model = RNNClassifier(vocab_size, embed_dim, hidden_dim, num_labels)
    elif model_type == "lstm":
        model = LSTMClassifier(vocab_size, embed_dim, hidden_dim, num_labels)
    elif model_type == "gru":
        model = GRUClassifier(vocab_size, embed_dim, hidden_dim, num_labels)
    elif model_type == "bilstm_rnn":
        model = BiLSTM_RNN_Classifier(vocab_size, embed_dim, hidden_dim, hidden_dim, num_labels)
    elif model_type == "bilstm_gru":
        model = BiLSTM_GRU_Classifier(vocab_size, embed_dim, hidden_dim, hidden_dim, num_labels)
    else:
        raise ValueError("Unknown model_type")

    print(f"\n[{model_type.upper()}] Training on {dataset_name}")
    model = train_rnn_style_model(model, train_loader, val_loader, num_epochs=epochs, lr=1e-3)

    val_acc, val_f1 = evaluate_model(model, val_loader)
    test_acc, test_f1 = evaluate_model(model, test_loader)
    print(f"[{model_type.upper()}] {dataset_name} -> val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, test_acc={test_acc:.4f}, test_f1={test_f1:.4f}")
    return val_acc, val_f1, test_acc, test_f1


In [ ]:
def run_transformer_with_metrics(
    dataset_name,
    model_name="distilbert-base-uncased",
    epochs=2,
    max_length=128,
    batch_size=32,
    lr=2e-5,
):
    train_loader, val_loader, test_loader, num_labels = create_dataloaders(
        dataset_name, max_length=max_length, batch_size=batch_size
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
    )
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        val_acc, val_f1 = evaluate_transformer(model, val_loader)
        print(f"[TRANSF {model_name}] {dataset_name}, epoch {epoch}: train_loss={total_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}")

    val_acc, val_f1 = evaluate_transformer(model, val_loader)
    test_acc, test_f1 = evaluate_transformer(model, test_loader)
    print(f"[TRANSF {model_name}] {dataset_name} -> val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, test_acc={test_acc:.4f}, test_f1={test_f1:.4f}")
    return val_acc, val_f1, test_acc, test_f1


In [ ]:
# 1) Build combined dataset from your uploaded TSVs + Hugging Face emotion datasets
import pandas as pd
import numpy as np
from datasets import load_dataset

local_paths = {
    "airline": "/content/drive/MyDrive/data/data/non_cuda/airline.tsv",
    "eco_news": "/content/drive/MyDrive/data/data/non_cuda/eco_news.tsv",
    "glo_warm": "/content/drive/MyDrive/data/data/non_cuda/glo_warm.tsv",
    "text_emo": "/content/drive/MyDrive/data/data/non_cuda/text_emo.tsv",
}

parts = []
for name, path in local_paths.items():
    df = pd.read_csv(path, sep="\t", encoding="latin1")
    df = df.rename(columns={c: c.strip() for c in df.columns})
    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError(f"{path} must contain 'text' and 'label' columns. Found: {df.columns.tolist()}")
    tmp = df[["text","label"]].copy()
    tmp["source"] = name
    parts.append(tmp)

hf_emotion = load_dataset("dair-ai/emotion")   # multi-emotion tweets
hf_tweet_eval = load_dataset("tweet_eval", "sentiment")  # 3-class sentiment

def hf_to_df(ds, text_col="text", label_col="label", source_name="hf"):
    rows = []
    for split in ds.keys():
        for ex in ds[split]:
            rows.append({"text": ex[text_col], "label": str(ex[label_col]), "source": source_name})
    return pd.DataFrame(rows)

parts.append(hf_to_df(hf_emotion, source_name="emotion_hf"))
parts.append(hf_to_df(hf_tweet_eval, source_name="tweet_eval"))

combined_df = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print("Combined shape:", combined_df.shape)
print(combined_df.source.value_counts())


Combined shape: (146764, 3)
source
tweet_eval    59899
text_emo      40000
emotion_hf    20000
airline       14640
eco_news       8000
glo_warm       4225
Name: count, dtype: int64


In [ ]:
# 2) Normalize labels, encode, and create stratified train/val/test splits
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

combined_df["text"] = combined_df["text"].astype(str)
combined_df["label"] = combined_df["label"].astype(str).str.strip().str.lower()

le_combined = LabelEncoder()
combined_df["label_id"] = le_combined.fit_transform(combined_df["label"])

X = combined_df["text"]
y = combined_df["label_id"]

# train 70%, val 15%, test 15% stratified
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Sizes:", len(X_train), len(X_val), len(X_test))
splits = {}          # reuse/replace previous
encoders = {}
splits["combined"] = {
    "train": (X_train.reset_index(drop=True), y_train.reset_index(drop=True)),
    "val": (X_val.reset_index(drop=True), y_val.reset_index(drop=True)),
    "test": (X_test.reset_index(drop=True), y_test.reset_index(drop=True))
}
encoders["combined"] = le_combined
print("Num classes (combined):", len(le_combined.classes_))


Sizes: 102734 22015 22015
Num classes (combined): 26


In [ ]:
# 3) Tokenizer + dataloader builder
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def create_dataloaders(dataset_name, max_length=128, batch_size=64):
    X_train, y_train = splits[dataset_name]["train"]
    X_val, y_val = splits[dataset_name]["val"]
    X_test, y_test = splits[dataset_name]["test"]

    def enc(texts):
        return tokenizer(list(texts), padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")

    train_enc = enc(X_train)
    val_enc = enc(X_val)
    test_enc = enc(X_test)

    train_ds = TextDataset(train_enc, y_train.to_numpy())
    val_ds = TextDataset(val_enc, y_val.to_numpy())
    test_ds = TextDataset(test_enc, y_test.to_numpy())

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    num_labels = len(np.unique(y_train))
    return train_loader, val_loader, test_loader, num_labels

# quick smoke test
train_loader, val_loader, test_loader, num_labels = create_dataloaders("combined", max_length=128, batch_size=64)
print("Num labels:", num_labels)


Device: cuda
Num labels: 26


In [ ]:
# 4) RNN/LSTM/GRU/BiLSTM+RNN/BiLSTM+GRU model definitions
import torch.nn as nn

vocab_size = tokenizer.vocab_size
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, num_labels)
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        _, h = self.rnn(x)
        h = h[-1]
        h = self.drop(h)
        return self.fc(h)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, num_labels)
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        _, (h, _) = self.lstm(x)
        h = h[-1]
        h = self.drop(h)
        return self.fc(h)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, num_labels)
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        _, h = self.gru(x)
        h = h[-1]
        h = self.drop(h)
        return self.fc(h)

class BiLSTM_RNN_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_bi, hidden_rnn, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.bilstm = nn.LSTM(embed_dim, hidden_bi, num_layers=1, batch_first=True, bidirectional=True)
        self.rnn = nn.RNN(2*hidden_bi, hidden_rnn, num_layers=1, batch_first=True)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_rnn, num_labels)
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        out, _ = self.bilstm(x)
        _, h = self.rnn(out)
        h = h[-1]
        h = self.drop(h)
        return self.fc(h)

class BiLSTM_GRU_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_bi, hidden_gru, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.bilstm = nn.LSTM(embed_dim, hidden_bi, num_layers=1, batch_first=True, bidirectional=True)
        self.gru = nn.GRU(2*hidden_bi, hidden_gru, num_layers=1, batch_first=True)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_gru, num_labels)
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        out, _ = self.bilstm(x)
        _, h = self.gru(out)
        h = h[-1]
        h = self.drop(h)
        return self.fc(h)


In [ ]:
# 5) Training/eval helpers for RNN-style models and wrapper to run experiments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_rnn(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids)
            all_logits.append(logits.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds = np.argmax(all_logits, axis=1)
    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, average="macro")
    return acc, f1, preds, all_labels

def train_rnn(model, train_loader, val_loader, epochs=3, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for ep in range(1, epochs+1):
        model.train()
        tot_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            opt.zero_grad()
            logits = model(input_ids)
            loss = crit(logits, labels)
            loss.backward()
            opt.step()
            tot_loss += loss.item()
        val_acc, val_f1, _, _ = evaluate_rnn(model, val_loader)
        print(f"Epoch {ep}: train_loss={tot_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}")
    return model

def run_rnn_with_metrics(dataset_name, model_type="lstm", epochs=3, batch_size=64, embed_dim=128, hidden_dim=128):
    train_loader, val_loader, test_loader, num_labels = create_dataloaders(dataset_name, max_length=128, batch_size=batch_size)
    vocab_size = tokenizer.vocab_size
    if model_type=="rnn":
        model = RNNClassifier(vocab_size, embed_dim, hidden_dim, num_labels)
    elif model_type=="lstm":
        model = LSTMClassifier(vocab_size, embed_dim, hidden_dim, num_labels)
    elif model_type=="gru":
        model = GRUClassifier(vocab_size, embed_dim, hidden_dim, num_labels)
    elif model_type=="bilstm_rnn":
        model = BiLSTM_RNN_Classifier(vocab_size, embed_dim, hidden_dim, hidden_dim, num_labels)
    elif model_type=="bilstm_gru":
        model = BiLSTM_GRU_Classifier(vocab_size, embed_dim, hidden_dim, hidden_dim, num_labels)
    else:
        raise ValueError("Unknown model type")
    model = train_rnn(model, train_loader, val_loader, epochs=epochs)
    val_acc, val_f1, _, _ = evaluate_rnn(model, val_loader)
    test_acc, test_f1, preds, labels = evaluate_rnn(model, test_loader)
    print(f"[{model_type.upper()}] {dataset_name} -> val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, test_acc={test_acc:.4f}, test_f1={test_f1:.4f}")
    return val_acc, val_f1, test_acc, test_f1


In [ ]:
# 6) Transformer (DistilBERT) training & eval (uses create_dataloaders)
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

def evaluate_transformer(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            all_logits.append(logits.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds = np.argmax(all_logits, axis=1)
    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, average="macro")
    return acc, f1, preds, all_labels

def run_transformer_with_metrics(dataset_name, model_name="distilbert-base-uncased", epochs=2, batch_size=32, max_length=128, lr=2e-5):
    train_loader, val_loader, test_loader, num_labels = create_dataloaders(dataset_name, max_length=max_length, batch_size=batch_size)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)
    opt = AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader)*epochs
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
    for ep in range(1, epochs+1):
        model.train()
        tot_loss = 0.0
        for batch in train_loader:
            opt.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            opt.step()
            sched.step()
            tot_loss += loss.item()
        val_acc, val_f1, _, _ = evaluate_transformer(model, val_loader)
        print(f"Epoch {ep}: train_loss={tot_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}, val_f1={val_f1:.4f}")
    val_acc, val_f1, _, _ = evaluate_transformer(model, val_loader)
    test_acc, test_f1, preds, labels = evaluate_transformer(model, test_loader)
    print(f"[TRANSFORMER {model_name}] {dataset_name} -> val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, test_acc={test_acc:.4f}, test_f1={test_f1:.4f}")
    return val_acc, val_f1, test_acc, test_f1


In [ ]:
models_to_run = [
    ("lstm", 50),
    ("gru", 50),
    ("bilstm_gru", 50),
    ("bilstm_rnn", 50),
]
exp_results = []

for mtype, epochs in models_to_run:
    print("\n========================")
    print("Model:", mtype)
    print("========================")
    v_acc, v_f1, t_acc, t_f1 = run_rnn_with_metrics("combined", model_type=mtype, epochs=epochs, batch_size=64)
    exp_results.append({"dataset":"combined", "model":mtype.upper(), "val_acc":v_acc, "val_f1":v_f1, "test_acc":t_acc, "test_f1":t_f1})

# Transformer
print("\n========================")
print("Model: DistilBERT")
print("========================")
v_acc, v_f1, t_acc, t_f1 = run_transformer_with_metrics("combined", model_name="distilbert-base-uncased", epochs=10, batch_size=32)
exp_results.append({"dataset":"combined", "model":"DistilBERT", "val_acc":v_acc, "val_f1":v_f1, "test_acc":t_acc, "test_f1":t_f1})

import pandas as pd
results_df = pd.DataFrame(exp_results)
print(results_df)



Model: lstm
Epoch 1: train_loss=2.3883, val_acc=0.2775, val_f1=0.0494
Epoch 2: train_loss=2.3716, val_acc=0.2767, val_f1=0.0513
Epoch 3: train_loss=2.3675, val_acc=0.2769, val_f1=0.0501
Epoch 4: train_loss=2.3610, val_acc=0.2750, val_f1=0.0528
Epoch 5: train_loss=2.3536, val_acc=0.2728, val_f1=0.0537
Epoch 6: train_loss=2.3480, val_acc=0.2739, val_f1=0.0533
Epoch 7: train_loss=2.3444, val_acc=0.2717, val_f1=0.0539
Epoch 8: train_loss=2.3433, val_acc=0.2702, val_f1=0.0526
Epoch 9: train_loss=2.3425, val_acc=0.2719, val_f1=0.0529
Epoch 10: train_loss=2.2759, val_acc=0.3426, val_f1=0.0912
Epoch 11: train_loss=1.6140, val_acc=0.4013, val_f1=0.1475
Epoch 12: train_loss=1.4599, val_acc=0.4145, val_f1=0.1586
Epoch 13: train_loss=1.3955, val_acc=0.4190, val_f1=0.1813
Epoch 14: train_loss=1.3531, val_acc=0.4375, val_f1=0.2059
Epoch 15: train_loss=1.3927, val_acc=0.4324, val_f1=0.2141
Epoch 16: train_loss=1.3098, val_acc=0.4528, val_f1=0.2322
Epoch 17: train_loss=1.2339, val_acc=0.4784, val_f1=

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1: train_loss=1.5609, val_acc=0.6382, val_f1=0.4103
Epoch 2: train_loss=0.9035, val_acc=0.6666, val_f1=0.4388
Epoch 3: train_loss=0.7748, val_acc=0.6681, val_f1=0.4491
Epoch 4: train_loss=0.6641, val_acc=0.6666, val_f1=0.4581
Epoch 5: train_loss=0.5553, val_acc=0.6621, val_f1=0.4552
